# DevSecOps CI/CD Pipeline with Automated Security Gates

**Purpose:** Demonstrate an automated security pipeline for a Python repository using GitHub Actions, Bandit, pip-audit, and Trivy.

## What this notebook does
1. Creates `security_scanner.py`, a small standard-library source checker.
2. Creates `.github/workflows/security.yml`, a GitHub Actions workflow that runs Python security checks and Trivy filesystem scanning.
3. Runs a local demonstration against a sample project folder.
4. Explains how to view the workflow results in GitHub Actions.

> Use this only on code and repositories you own or are authorized to assess. This is a learning project, not a guarantee that software is vulnerability-free.


## 1. Set up the project

Run the next cell from this notebook. It creates the scanner and workflow files in the current working directory. If you run it in a new folder, it will create the needed folders automatically.

In [ ]:
from pathlib import Path
import textwrap

ROOT = Path.cwd()
WORKFLOW_PATH = ROOT / ".github" / "workflows" / "security.yml"
SCANNER_PATH = ROOT / "security_scanner.py"

scanner_source = r"""
from pathlib import Path
import re
import sys
import json

# Intentionally basic indicators for an educational demo; not a full SAST engine.
RULES = [
    ("HIGH", "Possible hard-coded credential", re.compile(
        r"(?i)(password|passwd|api[_-]?key|secret|token)\s*=\s*['\"][^'\"]{6,}['\"]")),
    ("MEDIUM", "Use of eval()", re.compile(r"\beval\s*\(")),
    ("MEDIUM", "Use of shell=True in subprocess call", re.compile(r"shell\s*=\s*True")),
    ("LOW", "TLS certificate verification disabled", re.compile(r"verify\s*=\s*False")),
]

SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", "node_modules", ".ipynb_checkpoints"}
SOURCE_SUFFIXES = {".py"}

def scan_file(path: Path):
    findings = []
    try:
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    except OSError as exc:
        return [{"file": str(path), "line": 0, "severity": "LOW",
                 "rule": "File read error", "evidence": str(exc)}]
    for line_no, line in enumerate(lines, start=1):
        stripped = line.strip()
        if stripped.startswith("#"):
            continue
        for severity, rule, pattern in RULES:
            if pattern.search(line):
                findings.append({
                    "file": str(path),
                    "line": line_no,
                    "severity": severity,
                    "rule": rule,
                    "evidence": stripped[:180],
                })
    return findings

def scan_path(target: Path):
    if not target.exists():
        raise FileNotFoundError(f"Target does not exist: {target}")
    files = [target] if target.is_file() else [
        p for p in target.rglob("*")
        if p.is_file() and p.suffix in SOURCE_SUFFIXES
        and not any(part in SKIP_DIRS for part in p.parts)
    ]
    findings = []
    for path in files:
        findings.extend(scan_file(path))
    return findings

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Educational Python source security checker")
    parser.add_argument("target", nargs="?", default=".", help="Python file or project directory")
    parser.add_argument("--json", dest="json_path", help="Optional JSON output file")
    args = parser.parse_args()
    findings = scan_path(Path(args.target))
    counts = {level: sum(1 for item in findings if item["severity"] == level)
              for level in ("HIGH", "MEDIUM", "LOW")}
    print("DevSecOps Educational Source Scan")
    print(f"Target: {args.target}")
    print(f"Findings: {len(findings)} | HIGH: {counts['HIGH']} | MEDIUM: {counts['MEDIUM']} | LOW: {counts['LOW']}")
    for item in findings:
        print(f"[{item['severity']}] {item['file']}:{item['line']} - {item['rule']}")
        print(f"    Evidence: {item['evidence']}")
    if args.json_path:
        Path(args.json_path).write_text(json.dumps({"summary": counts, "findings": findings}, indent=2), encoding="utf-8")
        print(f"JSON report written to {args.json_path}")
    # Return non-zero for HIGH findings to support a simple local security gate.
    return 1 if counts["HIGH"] else 0

if __name__ == "__main__":
    sys.exit(main())
"""

workflow_yaml = """name: DevSecOps Security Pipeline

on:
  push:
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  python-security:
    name: Python security checks
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install project dependencies
        run: |
          python -m pip install --upgrade pip
          if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
          pip install bandit pip-audit

      - name: Run educational source scanner
        run: python security_scanner.py . --json security-scan-report.json

      - name: Run Bandit static analysis
        run: bandit -r . -x ./.git,./.venv,./venv -ll

      - name: Audit installed Python dependencies
        run: |
          if [ -f requirements.txt ] && [ -s requirements.txt ]; then
            pip-audit -r requirements.txt
          else
            echo "No pinned runtime dependencies found; skipping pip-audit dependency check."
          fi

      - name: Upload educational scan report
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: security-scan-report
          path: security-scan-report.json
          if-no-files-found: ignore

  trivy-filesystem:
    name: Trivy filesystem scan
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Scan repository with Trivy
        uses: aquasecurity/trivy-action@0.28.0
        with:
          scan-type: fs
          scan-ref: .
          format: table
          severity: HIGH,CRITICAL
          exit-code: "1"
"""

ROOT.mkdir(parents=True, exist_ok=True)
SCANNER_PATH.write_text(textwrap.dedent(scanner_source).strip() + "\n", encoding="utf-8")
WORKFLOW_PATH.parent.mkdir(parents=True, exist_ok=True)
WORKFLOW_PATH.write_text(workflow_yaml, encoding="utf-8")

print(f"Created: {SCANNER_PATH}")
print(f"Created: {WORKFLOW_PATH}")


## 2. Create a safe demo target

The following cell creates a tiny sample file with a deliberately unsafe placeholder assignment so you can see the scanner detect it. Do not put real passwords or API keys in source code. The placeholder is only for demonstration.

In [ ]:
from pathlib import Path
import subprocess, sys

demo_dir = Path("demo_app")
demo_dir.mkdir(exist_ok=True)
demo_file = demo_dir / "example.py"
demo_file.write_text(
    "# Demo-only finding; never use real credentials\n"
    "api_key = 'DEMO_ONLY_NOT_A_REAL_KEY'\n"
    "def calculate(value):\n"
    "    return eval(value)\n",
    encoding="utf-8",
)
print("Created demo target:", demo_file)


## 3. Run the local scanner

The scanner returns exit code `1` if it finds a HIGH-severity indicator. For the demonstration, that is expected because the sample intentionally contains a fake hard-coded key. Remove or fix the sample issue to get a clean result.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, str(SCANNER_PATH), str(demo_dir), "--json", "demo-report.json"],
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Scanner exit code:", result.returncode, "(1 is expected for the intentionally unsafe demo sample)")


## 4. Run checks locally

Install the optional CI tools from `requirements.txt`, then run:

```bash
python security_scanner.py .
python security_scanner.py . --json security-scan-report.json
bandit -r . -x ./.git,./.venv,./venv -ll
pip-audit -r requirements.txt
```

The notebook's scanner uses only the Python standard library. `bandit` and `pip-audit` are used by the GitHub Actions workflow for additional checks.

## 5. Enable the GitHub Actions pipeline

1. Push the generated `.github/workflows/security.yml` and `security_scanner.py` to a GitHub repository.
2. Open the repository's **Actions** tab.
3. Open **DevSecOps Security Pipeline** to see job results.
4. Inspect the `security-scan-report` artifact after a run.
5. The workflow is configured to fail on HIGH findings from the educational scanner, Bandit findings at the selected severity, pip-audit dependency findings, and HIGH/CRITICAL Trivy findings.

A failing pipeline is a security gate doing its job. Fix the issue and push again. Scan results can include false positives or miss issues, so review findings rather than treating tools as proof of security.

## Limitations

- The custom scanner uses simple text patterns and can produce false positives or miss complex vulnerabilities.
- Bandit focuses on Python code; Trivy checks repository files and detected vulnerabilities/misconfigurations within its supported scope.
- `pip-audit` requires dependencies to be specified in `requirements.txt`; for reproducible results, pin versions in real projects.
- SonarQube is not required for this runnable baseline. It can be added later if a SonarQube server or SonarCloud project is available.
